In [1]:
import os
import sys

In [8]:
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())
os.getcwd()

'c:\\dev\\predictive-maintenance-mlops'

In [9]:
import tempfile 
import joblib

from pathlib import Path
import time
import mlflow

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

from turbofan.data import RAW_DIR, add_rul, load_cmapss, split_by_engine
from turbofan.features import (
    FEATURE_SENSORS,
    RUL_CAP,
    WINDOW,
    add_rolling_features,
    cap_rul,
    get_feature_columns,
)

train = load_cmapss(RAW_DIR / "train_FD001.txt")
train = add_rul(train)
train = cap_rul(train)
df = add_rolling_features(train)

tr, val = split_by_engine(df)
print(tr["unit"].nunique(), "train engines,", val["unit"].nunique(), "validation engines")

c:\dev\predictive-maintenance-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


80 train engines, 20 validation engines


First Tracked Run

In [10]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("rul-fd001")

2026/10/24 14:45:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 14:45:11 INFO mlflow.store.db.utils: Updating database tables
2026/10/24 14:45:12 INFO mlflow.tracking.fluent: Experiment with name 'rul-fd001' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/dev/predictive-maintenance-mlops/mlruns/1', creation_time=1792878312798, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792878312798, lifecycle_stage='active', name='rul-fd001', tags={}, trace_location=None, workspace='default'>

In [11]:
with mlflow.start_run(run_name="hello-mlflow"):
    mlflow.log_param("note", "my first run")
    mlflow.log_metric("example_score", 42.0)

In [21]:
def run_experiment(run_name, columns, feature_set, n_estimators=100, max_depth=None, min_samples_leaf=1):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "feature_set": feature_set,
            "n_features": len(columns),
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "rul_cap": RUL_CAP,
            "window": WINDOW,
        }
        )

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1,
        )

        start = time.perf_counter()
        model.fit(tr[columns], tr["rul"])
        train_seconds = time.perf_counter() - start

        val_rmse = root_mean_squared_error(val["rul"], model.predict(val[columns]))

        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp) / "model.joblib"
            joblib.dump(model, path)
            size_mb = path.stat().st_size / 1000000

        mlflow.log_metrics({
            "val_rmse": val_rmse,
            "train_seconds": train_seconds,
            "model_size_mb": size_mb,
        })
        print(f"{run_name}: val_rmse={val_rmse:.2f}, size={size_mb:.1f} MB")

In [22]:
features_all = get_feature_columns()

run_experiment("raw-sensors", FEATURE_SENSORS, feature_set="raw")
run_experiment("engineered", features_all, feature_set="engineered")

for depth in [8, 12, 16]:
    run_experiment(f"engineered-depth{depth}", features_all, feature_set="engineered", max_depth=depth)

run_experiment("engineered-leaf5", features_all,
               feature_set="engineered", min_samples_leaf=5)

raw-sensors: val_rmse=17.13, size=92.7 MB
engineered: val_rmse=16.79, size=81.9 MB
engineered-depth8: val_rmse=16.47, size=3.6 MB
engineered-depth12: val_rmse=16.61, size=25.2 MB
engineered-depth16: val_rmse=16.75, size=54.8 MB
engineered-leaf5: val_rmse=16.68, size=18.9 MB


In [23]:
runs = mlflow.search_runs(
    experiment_names=["rul-fd001"],
    order_by=["metrics.val_rmse ASC"],
)

cols = ["tags.mlflow.runName", "params.max_depth", "params.min_samples_leaf", "metrics.val_rmse", "metrics.model_size_mb", "metrics.train_seconds"]
runs[cols]

,tags.mlflow.runName,params.max_depth,params.min_samples_leaf,metrics.val_rmse,metrics.model_size_mb,metrics.train_seconds
0,engineered-depth8,8,1,16.470863,3.625857,1.919653
1,engineered-depth12,12,1,16.613335,25.203105,2.973412
2,engineered-leaf5,None,5,16.684996,18.924993,3.017825
3,engineered-depth16,16,1,16.750527,54.818289,3.786702
4,engineered,None,1,16.792366,81.931185,4.283369
5,raw-sensors,None,1,17.134261,92.724529,1.146968
6,engineered-leaf5,None,5,NaN,18.924993,3.447568
7,engineered-depth16,16,1,NaN,54.818289,4.082242
8,engineered-depth12,12,1,NaN,25.203105,3.109314
9,engineered-depth8,8,1,NaN,3.625857,1.971604
